In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:
data = pd.read_csv('/content/reviews.csv', header=None, names = ('review','sentiment'), sep='\t')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31552 entries, 0 to 31551
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     31552 non-null  object
 1   sentiment  31551 non-null  object
dtypes: object(2)
memory usage: 493.1+ KB


In [ ]:
data.head()

In [ ]:
data.tail()

#Предобработка

In [ ]:
import re
import nltk
!pip install pymorphy3
from pymorphy3 import MorphAnalyzer
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
morph = MorphAnalyzer()
raw_reviews = data['review'].astype(str)

def clean_text(input_string):
  """
  Убираем левые символы, числа и приводим в нижний регистр
  """

  cleaned_string = re.sub(r'[^\w\s]', '', input_string)
  cleaned_string = re.sub(r'\d+', '', input_string)
  return cleaned_string.lower()

stop_words = set(stopwords.words('russian'))
lemma_cache = {}

def process_review_text(raw_input):
  """
  Лемматизация, стоп-слова, очистка
  """
  cleaned_text = clean_text(raw_input)
  word_tokens = word_tokenize(cleaned_text, language='russian')
  filtered_tokens = [token for token in word_tokens if token not in stop_words]

  processed_tokens = []
  for current_word in filtered_tokens:
    if current_word in lemma_cache:
      lemma = lemma_cache[current_word]
    else:
      lemma = morph.parse(current_word)[0].normal_form
      lemma_cache[current_word] = lemma
    processed_tokens.append(lemma)

  return ' '.join(processed_tokens)

data['review'] = raw_reviews.apply(process_review_text)
data.to_csv('clean.csv', index=False)

#Обучение модели

In [ ]:
data = pd.read_csv('/content/clean.csv', header=None, names=('review', 'sentiment'))
data = data.dropna(subset=['review'])
text_reviews = data['review'].values
y = data['sentiment'].values
text_reviews = [str(review) for review in text_reviews]

In [ ]:
filtered_reviews = []
filtered_y = []

for review, label in zip(text_reviews, y):
  if review.strip() != '':
    filtered_reviews.append(review)
    filtered_y.append(label)

text_reviews = filtered_reviews
y = np.array(filtered_y)

In [ ]:
pipeline = make_pipeline(
  CountVectorizer(),
  LogisticRegression(n_jobs=-1, random_state=2)
)

pipeline.fit(text_reviews, y)

Pipeline(steps=[('countvectorizer', CountVectorizer()),
                ('logisticregression',
                 LogisticRegression(n_jobs=-1, random_state=17))])

In [ ]:
print("Точность на обучающей выборке:", round(pipeline.score(text_reviews, y), 3))

Точность на обучающей выборке: 0.966


In [ ]:
param_grid = {'logisticregression__C': np.logspace(-2, 3, 15)}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)
grid = GridSearchCV(pipeline, param_grid, cv=skf, n_jobs=-1)
grid.fit(text_reviews, y)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=17, shuffle=True),
             estimator=Pipeline(steps=[('countvectorizer', CountVectorizer()),
                                       ('logisticregression',
                                        LogisticRegression(n_jobs=-1,
                                                           random_state=17))]),
             n_jobs=-1,
             param_grid={'logisticregression__C': array([1.00000000e-02, 2.27584593e-02, 5.17947468e-02, 1.17876863e-01,
       2.68269580e-01, 6.10540230e-01, 1.38949549e+00, 3.16227766e+00,
       7.19685673e+00, 1.63789371e+01, 3.72759372e+01, 8.48342898e+01,
       1.93069773e+02, 4.39397056e+02, 1.00000000e+03])})

In [ ]:
print("Лучшие параметры:", grid.best_params_)
print("Лучшее качество по кросс-валидации:", grid.best_score_)

In [ ]:
coef = grid.best_estimator_.named_steps['logisticregression'].coef_.ravel()
feature_names = grid.best_estimator_.named_steps['countvectorizer'].get_feature_names_out()

print("Длина coef:", len(coef))
print("Длина feature_names:", len(feature_names))


In [ ]:
print("Длина coef:", len(coef))
print("Длина feature_names:", len(feature_names))
print("coef:", coef)
print("feature_names:", feature_names)

In [ ]:
print("Минимальный coef:", np.min(coef))
print("Максимальный coef:", np.max(coef))

In [ ]:
n_top_features = min(25, len(coef)//2)
print(n_top_features)

In [ ]:
coef = grid.best_estimator_.named_steps['logisticregression'].coef_.ravel()
feature_names = grid.best_estimator_.named_steps['countvectorizer'].get_feature_names_out()

print("Длина coef:", len(coef))
print("Длина feature_names:", len(feature_names))
print("coef:", coef)
print("feature_names:", feature_names)

num_features_to_show = min(25, len(coef) // 2)

indices_for_positive_impact = np.argsort(coef)[-num_features_to_show:]
indices_for_negative_impact = np.argsort(coef)[:num_features_to_show]

combined_important_indices = np.hstack([indices_for_negative_impact, indices_for_positive_impact])

print("combined_important_indices перед фильтрацией:", combined_important_indices)

combined_important_indices = combined_important_indices[combined_important_indices < len(feature_names)]

print("combined_important_indices после фильтрации:", combined_important_indices)

if len(combined_important_indices) == 0:
  print("Нет достаточно интересных признаков для отображения.")
else:
  actual_display_count = min(len(combined_important_indices), 2 * num_features_to_show)

  plt.figure(figsize=(15, 5))
  bar_colors = ["red" if c < 0 else "blue" for c in coef[combined_important_indices][:actual_display_count]]
  plt.bar(np.arange(actual_display_count), coef[combined_important_indices][:actual_display_count], color=bar_colors)
  plt.xticks(np.arange(actual_display_count), feature_names[combined_important_indices][:actual_display_count], rotation=60, ha="right")
  plt.title("Топ слов по коэффициентам модели")
  plt.show()


In [ ]:
n_top_features = 25
positive_coefficients = np.argsort(coef)[-n_top_features:]
negative_coefficients = np.argsort(coef)[:n_top_features]
interesting_coefficients = np.hstack([negative_coefficients, positive_coefficients])

plt.figure(figsize=(15, 5))
colors = ["red" if c < 0 else "blue" for c in coef[interesting_coefficients]]
plt.bar(np.arange(2 * n_top_features), coef[interesting_coefficients], color=colors)
plt.xticks(np.arange(2 * n_top_features), feature_names[interesting_coefficients], rotation=60, ha="right")
plt.title("Топ-слова по коэффициентам")
plt.show()